## Overview

CNN-GRU Setup for prediction out of 2D timeseries data

finding out of impact of training on "wrong" year of the saison

----
## Data:

-2D Space - Timeseries

-predicting 1 feature out of 7 variables

-Forecasting 1 timestep (not the following)

-------
Peter Resch, 3.6.

next step: get the window method in the dataset class to the agreed format

In [85]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np

import seaborn as sns

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split

import xarray as xr

rSeed=42

SavePlots = False

## Loading Data

In [86]:
!cd ../../../datasets/small_grid/ && ls -l

total 31444
-rw-rw-r-- 1 peter peter 9715104 May 28 11:56 grid-timeseries_sel_vars_2020.grib
-rw-rw-r-- 1 peter peter 6221476 May 29 16:56 grid-timeseries_sel_vars_2020.grib.nc
-rw-rw-r-- 1 peter peter 5680416 May 25 14:57 grid-timeseries_sel_vars_2025_summermonths.grib
-rw-rw-r-- 1 peter peter 3638732 May 29 16:57 grid-timeseries_sel_vars_2025_summermonths.grib.nc
-rw-rw-r-- 1 peter peter 4008144 May 27 15:31 grid-timeseries_sel_vars_2025_wintermonths.grib
-rw-rw-r-- 1 peter peter 2568236 May 29 16:59 grid-timeseries_sel_vars_2025_wintermonths.grib.nc
-rw-rw-r-- 1 peter peter  212352 May 25 14:58 grid-timeseries_sel_vars_2026_april.grib
-rw-rw-r-- 1 peter peter  138368 May 29 16:59 grid-timeseries_sel_vars_2026_april.grib.nc
drwxrwxr-x 2 peter peter    4096 Jun  3 14:11 new


In [ ]:
year="2020"
ending = ".grib.nc"
filename="../../../datasets/small_grid/grid-timeseries_sel_vars_"+year+ending

# Open the NetCDF files using xarray
ds={}
ds[year]= xr.open_dataset(filename)
ds1=ds[year]
ds["2021"]=ds1
ds.keys()

#combining the dict datasets into one xarray dataset
ds_combined = xr.concat([ds[year] for year in ds.keys()], dim='time')
ds_combined

ds_spring = ds_combined.sel(time=ds_combined['time.season'] == 'MAM')
ds_summer = ds_combined.sel(time=ds_combined['time.season'] == 'JJA')
ds_autumn = ds_combined.sel(time=ds_combined['time.season'] == 'SON')
ds_winter = ds_combined.sel(time=ds_combined['time.season'] == 'DJF')

ds=ds.sel(time=ds['time.season'] == 'MAM')

ds_test=ds.sel(time=ds['time.season'] == 'JJA')

In [117]:
class TimeseriesDatasetPipeline:
    """Pipeline to convert xarray Dataset to PyTorch Dataset with optional scaling."""
    
    def __init__(self, ds, scaler_set=None,lag_selection=None, forecast_horizon=24,forecast_hours=1,forecast_var=5):
        """
        Initialize the pipeline.
        
        Args:
            ds: xarray.Dataset with dimensions (time, lat, lon)
            scaler: sklearn scaler object (optional, e.g., StandardScaler)
            forecast_lag: number of past timesteps to use as input
            forecast_horizon: number of future timesteps to predict
        """
        self.scaler_set = [] if scaler_set is None else scaler_set
        self.ds = ds
        self.lag_selection = lag_selection
        self.forecast_horizon = forecast_horizon
        self.forecast_var = forecast_var
        self.forecast_hours = forecast_hours
        self.data = None
        self.scaled_data = None
        self._prepare_data()
        if self.forecast_var >= self.data.shape[1]:
            raise ValueError(f"forecast_var index {self.forecast_var} is out of bounds for data with {self.data.shape[1]} variables.")
    
    def get_input_shape(self):
        """Return the shape of the input data (time, variables, space)."""
        return self.ds.to_array().values.shape if self.ds is not None else None

    def _prepare_data(self):
        """Convert xarray to numpy and flatten spatial dimensions."""
        # Stack lat and lon into single spatial dimension
        stacked = self.ds.to_array()#.stack(space=('lat', 'lon'))  #(variables, time, space)
        self.data = stacked.values.transpose(1, 0, 2,3)             #(time, variables, space)
        print("Data prepared with shape:", self.data.shape)
        self._process_data()

    def _process_data(self):
        """Apply each scaler to the corresponding variable."""
        sh= self.data.shape
        #print("Process data with shape:", sh)
        time, vars_, lat,lon = sh
        #print(f"Processing data: time={time}, vars={vars_}, space={lat}x{lon}")

        scaled = np.zeros_like(self.data)
        for i in range(vars_):
            reshaped = self.data[:, i, :].reshape(-1, 1)  # (time*space, 1)
            #print("reshapded to shape:", reshaped.shape)

            if len(self.scaler_set) <= i or self.scaler_set[i] is None:
                sc=sklearn.preprocessing.StandardScaler().fit(reshaped)
                self.scaler_set.append(sc)
            scaled[:, i, :] = self.scaler_set[i].transform(reshaped).reshape(time, lat, lon)
        self.scaled_data = scaled

    def set_scaler_set(self, scaler_set):
        """Set an array of scalers (one per variable) and reprocess data."""
        self.scaler_set = scaler_set
        self._process_data()

    def get_scaler_set(self):
        """Return the list/array of scalers."""
        return self.scaler_set

    def create_windowed_dataset(self):
        """Create X and y using moving window method."""
        data = self.scaled_data
        if self.lag_selection:
            max_lag = max(self.lag_selection)+1
        else:
            raise ValueError("forecast_lag_selection must be provided.")
        X, y = [], []
        for i in range(len(data) - max_lag - self.forecast_horizon + 1):
            """if self.lag_selection == "logarithmic":
                #use 72,48,24,12,6,3,1 hours as input
                lag_selection = [72,48,24,12,6,3,1]
                lag_selection = [72-t for t in lag_selection]#converting to index selection
                print(f"Using logarithmic lag selection: {lag_selection}")
            else:
                raise ValueError(f"Unsupported lag_selection: {self.lag_selection}")
                #lag_selection = np.array(range(self.forecast_lag),)"""
            #print(f"\nforecast horizon: {self.forecast_horizon}, forecast var: {self.forecast_var}, lag selection: {self.lag_selection}")
            
            #print("Original data shape:", data.shape)
            data_filtered = []
            for t in self.lag_selection:
                if i+t >= len(data):
                    print(f"Skipping lag {t} at index {i} due to out of bounds.")
                    continue
                data_filtered.append(data[i+t,:,:])
            xd = np.array(data_filtered)  # (lag_selection, variables, space)
            #print("data_filtered:", getattr(xd, "shape", None))
            
            #xd=np.delete(data_filtered, self.forecast_var, axis=1)# Remove target variable from input
            #print("xd:", getattr(xd, "shape", None), type(xd),"\n")
            X.append(xd)
            yd=data[i+self.forecast_horizon,self.forecast_var,:]  # Only target variable
            yd.reshape
            y.append(yd)
            #print("yd:", getattr(yd, "shape", None), type(yd))

            #return test_end

        X = np.array(X)  # (samples, lag-1, space)
        y = np.array(y)  # (samples, horizon, space)
        print("X:", getattr(X, "shape", None),type(X))
        print("y:",getattr(y, "shape", None), type(y))
        #reshape y to (samples,1,space) for univariate prediction
        y=y.reshape(y.shape[0], 1, y.shape[1], y.shape[2])
        #print("y reshaped",y)
        return X,y
    
    def to_pytorch_dataset(self):
        """Convert to PyTorch Dataset."""
        X, y = self.create_windowed_dataset()
        return TimeseriesPyTorchDataset(X, y)

    def to_pytorch_dataloader(self, batch_size=32, shuffle=True):
        """Convert to PyTorch DataLoader."""
        dataset = self.to_pytorch_dataset()
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


class TimeseriesPyTorchDataset(Dataset):
    """PyTorch Dataset for timeseries data."""
    
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [118]:
#Testing
pipeline = TimeseriesDatasetPipeline(ds1, lag_selection=[0, 24, 48, 60, 66, 69, 71], forecast_horizon=1, forecast_var=5)
X,y=pipeline.create_windowed_dataset()

#X[0,0,0,:].shape#25
#X[0,0,0]
X.shape
y.shape



Data prepared with shape: (8784, 7, 5, 5)
X: (8712, 7, 7, 5, 5) <class 'numpy.ndarray'>
y: (8712, 5, 5) <class 'numpy.ndarray'>


(8712, 1, 5, 5)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

batch_size=64
lag_selection=[0, 24, 48, 60, 66, 69, 71]

validate_ratio=0.20
test_ratio=0
train_ratio=1-validate_ratio-test_ratio

n_total = len(ds.time)
n_test = int(n_total * test_ratio)
n_validate = int(n_total * validate_ratio)
n_train = n_total - n_test - n_validate

#select the data randomly for training and validation by making a random list of indices for training and validation

#ds_test=ds.isel(time=slice(-n_test, None))  # Last n_test time steps for testing
ds_validate=ds.isel(time=slice(-n_test - n_validate, -n_test))  # Next n_validate time steps for validation
ds_train=ds.isel(time=slice(0, n_train))  # All but last n_test + n_validate for training




#go through 






pipeline_train= TimeseriesDatasetPipeline(ds_train, lag_selection=lag_selection, forecast_horizon=1, forecast_var=5)
dataloader_train = pipeline_train.to_pytorch_dataloader(batch_size=batch_size, shuffle=True)
scaler_set_train = pipeline_train.get_scaler_set()

pipeline_validate = TimeseriesDatasetPipeline(ds_validate, lag_selection=lag_selection, forecast_horizon=1, forecast_var=5, scaler_set=scaler_set_train)
dataloader_validate= pipeline_validate.to_pytorch_dataloader(batch_size=batch_size, shuffle=True)

pipeline_test = TimeseriesDatasetPipeline(ds_test, lag_selection=lag_selection, forecast_horizon=1, forecast_var=5, scaler_set=scaler_set_train)
dataloader_test = pipeline_test.to_pytorch_dataloader(batch_size=batch_size, shuffle=False)


"""
#Histogram of first variable scaled and unscaled for training data

for i in range(7):
    #plot histogram of first variable scaled and unscaled
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.hist(pipeline.data[:, i, :].flatten(), bins=50, alpha=0.7, label='Unscaled')
    plt.title(f'Unscaled Variable {i+1}')
    plt.subplot(1, 2, 2)
    plt.hist(pipeline.scaled_data[:, i, :].flatten(), bins=50, alpha=0.7, label='Scaled')
    plt.title(f'Scaled Variable {i+1}')
    plt.tight_layout()
"""

Data prepared with shape: (6150, 7, 5, 5)
X: (6078, 7, 7, 5, 5) <class 'numpy.ndarray'>
y: (6078, 5, 5) <class 'numpy.ndarray'>
Data prepared with shape: (1317, 7, 5, 5)
X: (1245, 7, 7, 5, 5) <class 'numpy.ndarray'>
y: (1245, 5, 5) <class 'numpy.ndarray'>
Data prepared with shape: (1317, 7, 5, 5)
X: (1245, 7, 7, 5, 5) <class 'numpy.ndarray'>
y: (1245, 5, 5) <class 'numpy.ndarray'>


"\n#Histogram of first variable scaled and unscaled for training data\n\nfor i in range(7):\n    #plot histogram of first variable scaled and unscaled\n    plt.figure(figsize=(6, 3))\n    plt.subplot(1, 2, 1)\n    plt.hist(pipeline.data[:, i, :].flatten(), bins=50, alpha=0.7, label='Unscaled')\n    plt.title(f'Unscaled Variable {i+1}')\n    plt.subplot(1, 2, 2)\n    plt.hist(pipeline.scaled_data[:, i, :].flatten(), bins=50, alpha=0.7, label='Scaled')\n    plt.title(f'Scaled Variable {i+1}')\n    plt.tight_layout()\n"

In [120]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


hidden_size=16

class CNN_GRU(nn.Module):
    def __init__(self,in_channels=7,hidden_size=16,lat_size=5,lon_size=5):
        super().__init__()

        #compressing space
        def double_conv(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )
        self.conv1 = double_conv(in_channels, 16)
        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))
        self.conv2 = double_conv(16, 32)

        #compressing time
        self.gru = nn.GRU(input_size=32,hidden_size=32*25, num_layers=1,dropout=0.2,batch_first=True)
        
        #expanding space
        self.deconv1 = nn.ConvTranspose2d(32, 16, kernel_size=3, padding=1)
        self.deconv2 = nn.ConvTranspose2d(16, 1, kernel_size=3, padding=1)


    def forward(self, x):
        #print("starting forward:",x.shape)
        x_gru=[]
        #recognize patterns of the spatial data with CNN
        for time in range(x.shape[1]):
            #print(time)
            cnn_in=x[:, time, :, :]
            #print("time,cnn_in.shape:",time,cnn_in.shape)
            cnn_in=self.conv1(cnn_in)
            #print("after conv1:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = self.conv2(cnn_in)
            #print("after conv2:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = cnn_in.view(cnn_in.size(0), -1)  # Flatten for GRU input
            #print("after flatten:",cnn_in.shape)
            x_gru.append(cnn_in)
        x_gru = torch.stack(x_gru, dim=1)
        #print("shaped for GRU:",x_gru.shape)#(batch, time, convoluted features with space)

        #decoding the temporal patterns with GRU
        x,_ = self.gru(x_gru)#x:(batch, time, hidden_size)
        x = x[:,-1,:].view(x.shape[0],32,5,5)##(batch, last hidden_size,lat_size,lon_size)
        #print("after GRU:",x.shape)
        x=self.deconv1(x)
        #print("after deconv1:",x.shape)
        x=self.deconv2(x)
        #print("after deconv2:",x.shape)
        return x# 


cnn_gru_model=CNN_GRU().to(device)
print(cnn_gru_model)

# Loss function and optimizer
loss_fcn = nn.MSELoss()
optimizer_cnn_gru = AdamW(cnn_gru_model.parameters(), lr=1e-3, weight_decay=1e-4)

Using cpu device
CNN_GRU(
  (conv1): Sequential(
    (0): Conv2d(7, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (gru): GRU(32, 800, batch_first=True, dropout=0.2)
  (deconv1): ConvTranspose2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (deconv2): ConvTranspose2d(16, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


/home/peter/anaconda3/envs/appmlenv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [128]:
def train(dataloader, model, loss_fn, optimizer,device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #print(X.shape, y.shape)
        X, y = X.to(device), y.to(device)
        #print(X.shape, y.shape)

        # Compute prediction error
        pred = model(X)#.squeeze()
        #print(pred)#.shape)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"Train Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return loss.item()



def test(dataloader, model, loss_fn,device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            test_loss += loss_fn(predictions, y).item()

            all_predictions.append(predictions.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())

    test_loss /= num_batches
    y_pred = np.concatenate(all_predictions)#[:,:,0]
    y_true = np.concatenate(all_targets)#[:,:,0]
    #print(f"y_true shape: {y_true.shape}, y_pred shape: {y_pred.shape}")


    # Flatten everything to 2D: (samples, features)
    y_true_flat = y_true.reshape(y_true.shape[0], -1)
    y_pred_flat = y_pred.reshape(y_pred.shape[0], -1)

    #mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    #rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    r2 = sklearn.metrics.r2_score(y_true_flat, y_pred_flat)
    print(f"Test Error:\n R2: {r2:>8f}, Avg loss: {test_loss:>8f} \n")
    return test_loss




In [ ]:
epochs = 3

train_losses=[]
val_losses=[]
test_losses=[]

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_losses.append(train(dataloader_train, cnn_gru_model, loss_fcn, optimizer_cnn_gru,device))
    val_losses.append(test(dataloader_validate, cnn_gru_model, loss_fcn,device)) 
    test_losses.append(test(dataloader_test, cnn_gru_model, loss_fcn,device)) 

train_losses=np.array(train_losses)
val_losses=np.array(val_losses)
test_losses=np.array(test_losses)

print("Done!")


Epoch 1
-------------------------------


Train Loss: 0.156498  [   64/ 6078]
Test Error:
 R2: 0.764665, Avg loss: 0.186371 

Epoch 2
-------------------------------
Train Loss: 0.133132  [   64/ 6078]


KeyboardInterrupt: 

## Optimizing - Hyperparametertuning

## saving trained model

In [ ]:
import os

season="summer"
os.makedirs("models", exist_ok=True)
save_path = f"models/cnn_gru_{season}.pth"

torch.save({
    "model_state_dict": cnn_gru_model.state_dict(),
    "optimizer_state_dict": optimizer_cnn_gru.state_dict(),
    "scaler_set": scaler_set_train,
    "lag_selection": lag_selection,
    "epoch": epochs
}, save_path)

print(f"Saved model to {save_path}")

Saved model to models/cnn_gru_summer.pth


In [ ]:
# plot for performance for each epoch - loss vs epoch

fig, ax = plt.subplots()
ax.plot(train_losses, label='Train Loss')
ax.plot(val_losses, label='Validate Loss')
ax.plot(test_losses, label='Test Loss')
ax.set_xlabel('Epoch')  
ax.set_ylabel('Loss')
ax.legend()
plt.show()



In [ ]:
# save train and test losses to csv
import pandas as pd

model_name = f"cnn_gru_{season}"

df = pd.DataFrame({'Train Loss': train_losses, 'Validate Loss': val_losses, 'Test Loss': test_losses})
df.to_csv(f'losses_{model_name}.csv', index=False)

## retrieve saved model

In [ ]:
# Load the saved model
checkpoint = torch.load(save_path)

# Create a new model instance
cnn_gru_model_loaded = CNN_GRU().to(device)

# Load the saved state
cnn_gru_model_loaded.load_state_dict(checkpoint["model_state_dict"])

# Optionally load other components
loaded_scaler_set = checkpoint["scaler_set"]
loaded_lag_selection = checkpoint["lag_selection"]
loaded_epoch = checkpoint["epoch"]

print(f"Model loaded from {save_path}")
print(f"Training stopped at epoch: {loaded_epoch}")
print(f"Lag selection: {loaded_lag_selection}")

# Set model to evaluation mode for inference
cnn_gru_model_loaded.eval()